<a href="https://colab.research.google.com/github/masum-mir/machine-learning-journey/blob/main/MissingValueHandling%26CustomCategoricalEncodingTechnique.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Missing Value Handling &             Custom Categorical Encoding Technique


In [57]:
import os
DATA_DIR = '/content/drive/MyDrive/CSE475/dataset/diabetes_dataset_missing_value.csv'
OUT_DIR  = '/content/drive/MyDrive/CSE475/dataset/plot'
if not os.path.exists(DATA_DIR):
    DATA_DIR = 'diabetes_dataset_missing_value.csv'; OUT_DIR = 'plots'
os.makedirs(OUT_DIR, exist_ok=True)
print("DATA_DIR:", DATA_DIR, "\nOUT_DIR :", OUT_DIR)

DATA_DIR: /content/drive/MyDrive/CSE475/dataset/diabetes_dataset_missing_value.csv 
OUT_DIR : /content/drive/MyDrive/CSE475/dataset/plot


In [50]:
import numpy as np, pandas as pd, warnings, time
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, BayesianRidge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor,
                              HistGradientBoostingRegressor, AdaBoostRegressor)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder

plt.rcParams.update({"figure.dpi":110,"axes.spines.top":False,"axes.spines.right":False})


## 1 · Load, encode categoricals, and separate data


In [51]:
df = pd.read_csv(DATA_DIR)
CAT = ["gender","smoking_history"];
BIN = ["hypertension","heart_disease"]
print("Shape:", df.shape)

# Step 1: Label Encoding (categorical -> numeric), NaN preserved
# encoders = {}
# E = df.copy()
# for c in CAT:
#     le = LabelEncoder()
#     mask = E[c].notna()
#     E.loc[mask, c] = le.fit_transform(E.loc[mask, c])
#     encoders[c] = le
custom_codes = {}   # category -> encoded numeric value
custom_inv   = {}   # encoded numeric value -> category (for decode, nearest match)

E = df.copy()
for c in CAT:
    valid = E[c].dropna()

    # ordinal rank
    sorted_cats = sorted(valid.unique())
    ordinal_rank = {cat: i for i, cat in enumerate(sorted_cats)}

    # frequency proportion of each category
    freq_prop = valid.value_counts(normalize=True).to_dict()

    # combine: ordinal_rank + frequency_proportion
    mapping = {}
    for cat in sorted_cats:
        mapping[cat] = ordinal_rank[cat] + freq_prop[cat]

    inv_mapping = {v: k for k, v in mapping.items()}

    custom_codes[c] = mapping
    custom_inv[c]   = inv_mapping

    E[c] = E[c].map(mapping)


E = E.astype(float)

# Step 2: separate complete and missing rows
complete = E.dropna().reset_index(drop=True)
missing  = E[E.isna().any(axis=1)]
cols_with_missing = [c for c in E.columns if E[c].isna().any()]
print(f"Complete rows: {len(complete)} | rows with missing: {len(missing)}")
print("Columns to impute:", cols_with_missing)

Shape: (199, 9)
Complete rows: 136 | rows with missing: 63
Columns to impute: ['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history', 'bmi', 'HbA1c_level', 'blood_glucose_level']


## 2 · Define the 15 prediction techniques

In [52]:
class ModeRegressor(BaseEstimator, RegressorMixin):
    # predicts the training mode (ignores X) - a categorical baseline
    def fit(self, X, y): self.m_ = pd.Series(np.asarray(y)).mode().iloc[0]; return self
    def predict(self, X): return np.full(len(X), self.m_)

def make_techniques():
    t = {
        "Mean":          DummyRegressor(strategy="mean"),
        "Median":        DummyRegressor(strategy="median"),
        "Mode":          ModeRegressor(),
        "LinearReg":     make_pipeline(StandardScaler(), LinearRegression()),
        "Ridge":         make_pipeline(StandardScaler(), Ridge()),
        "Lasso":         make_pipeline(StandardScaler(), Lasso()),
        "BayesianRidge": make_pipeline(StandardScaler(), BayesianRidge()),
        "DecisionTree":  DecisionTreeRegressor(max_depth=6, random_state=0),
        "RandomForest":  RandomForestRegressor(n_estimators=80, random_state=0),
        "ExtraTrees":    ExtraTreesRegressor(n_estimators=80, random_state=0),
        "GradientBoost": GradientBoostingRegressor(random_state=0),
        "HistGBM":       HistGradientBoostingRegressor(random_state=0),
        "AdaBoost":      AdaBoostRegressor(random_state=0),
        "KNN":           make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=5)),
        "SVR":           make_pipeline(StandardScaler(), SVR()),
    }
    return t

print("Number of techniques:", len(make_techniques()))

Number of techniques: 15


## 3 · Per-Feature Model Selection using MSE

In [53]:
N_SPLITS = 5
tech_names = list(make_techniques().keys())
mse_table = pd.DataFrame(index=tech_names, columns=cols_with_missing, dtype=float)

t0 = time.time()
for col in cols_with_missing:
    feats = [c for c in complete.columns if c != col]
    accum = {name: [] for name in tech_names}
    for sp in range(N_SPLITS):
        Xtr, Xte, ytr, yte = train_test_split(complete[feats], complete[col], test_size=0.2, random_state=sp)
        for name, est in make_techniques().items():
            try:
                est.fit(Xtr, ytr)
                accum[name].append(mean_squared_error(yte, est.predict(Xte)))
            except Exception:
                accum[name].append(np.nan)
    for name in tech_names:
        mse_table.loc[name, col] = np.nanmean(accum[name])



### 4.1 · Best technique per column

In [54]:
best_per_col = {col: mse_table[col].idxmin() for col in cols_with_missing}
summary = pd.DataFrame({
    "best_technique":[best_per_col[c] for c in cols_with_missing],
    "best_MSE":[mse_table.loc[best_per_col[c], c] for c in cols_with_missing],
}, index=cols_with_missing)
print("Technique for each column:")
summary.round(3)

Technique for each column:


,best_technique,best_MSE
gender,Mean,0.127
age,KNN,412.734
hypertension,Mean,0.048
heart_disease,Ridge,0.065
smoking_history,AdaBoost,3.261
bmi,SVR,32.666
HbA1c_level,LinearReg,0.942
blood_glucose_level,LinearReg,2061.342


## 5 · Fill the real missing values with each column's winning technique  

In [60]:
# temporary scaffold: every cell mean-filled, used only to provide features for prediction
scaffold = E.copy()
for c in E.columns:
    scaffold[c] = scaffold[c].fillna(complete[c].mean())

completed = E.copy()
for col in cols_with_missing:
    feats = [c for c in E.columns if c != col]
    model = make_techniques()[best_per_col[col]]
    model.fit(complete[feats], complete[col])
    rows_missing = E[col].isna()
    preds = model.predict(scaffold.loc[rows_missing, feats])
    completed.loc[rows_missing, col] = preds

# decode categoricals back to labels
for c in BIN:  completed[c] = completed[c].round().clip(0,1).astype(int)
# for c in CAT:
#     n_classes = len(encoders[c].classes_)
#     completed[c] = completed[c].round().clip(0, n_classes-1).astype(int)
#     completed[c] = encoders[c].inverse_transform(completed[c])
for c in CAT:
    valid_values = np.array(sorted(custom_inv[c].keys()), dtype=float)
    completed[c] = completed[c].apply(
        lambda v: custom_inv[c][valid_values[np.argmin(np.abs(valid_values - v))]]
    )

completed.head(50)

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.190000,6.600000,140.000000,0.0
1,Female,54.0,0,0,No Info,27.320000,6.600000,80.000000,0.0
2,Male,28.0,0,0,never,27.320000,5.700000,158.000000,0.0
3,Female,36.0,0,0,current,23.450000,5.000000,155.000000,0.0
4,Male,76.0,1,1,current,20.140000,4.800000,155.000000,0.0
5,Female,20.0,0,0,never,27.320000,6.600000,85.000000,0.0
6,Female,44.0,0,0,never,19.310000,6.500000,200.000000,1.0
7,Female,79.0,0,0,No Info,23.860000,5.700000,85.000000,0.0
8,Male,42.0,0,0,never,33.640000,4.800000,145.000000,0.0
9,Female,32.0,0,0,never,27.320000,5.000000,100.000000,0.0
